# 5장. 알고리즘 연습 (시간복잡도 · 자료구조 · 이분탐색 · BFS · 힙 · DP)

PDF 5장 요약을 손으로 확인하는 정도만 담았어요. 유형별 문제는 `코딩테스트_준비.pdf`로 이어서 해요.
각 문제는 **실무와 연결**돼요: DB 인덱스 = B-Tree(이분탐색의 확장), 캐시 = 해시, BM25 상위 k개 = 힙.

In [ ]:
# ▶ 먼저 이 셀을 실행하세요 (채점 도우미)
import os
STRICT = os.environ.get("PRACTICE_STRICT") == "1"
___ = None  # 빈칸 표시. 이 자리를 채우면 돼요

def check(name, cond, hint=""):
    """cond가 참이면 ✅, 아니면 ❌ (cond는 값이나 lambda)"""
    try:
        ok = bool(cond() if callable(cond) else cond)
    except Exception as e:
        ok, hint = False, f"{type(e).__name__}: {e}"
    print(("✅ " if ok else "❌ ") + name + ("" if ok or not hint else f"  → {hint}"))
    if STRICT and not ok:
        raise AssertionError(name)

In [ ]:
import time, random, bisect, heapq
from collections import deque, OrderedDict
random.seed(0)

## 5-1. 시간복잡도

In [ ]:
# 빠른 것 → 느린 것 순서로
order = [___]          # "O(n)", "O(1)", "O(n²)", "O(log n)", "O(n log n)" 정렬
# n = 100,000 일 때 O(n²)은 대략 몇 번 연산? (정수)
ops_n2 = ___
# 다음 연산의 평균 시간복잡도
complexity = {
    "list에서 x in lst":      ___,
    "set에서 x in s":         ___,
    "정렬":                   ___,
    "정렬된 리스트에서 bisect": ___,
    "heapq.heappush":         ___,
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
order = ["O(1)", "O(log n)", "O(n)", "O(n log n)", "O(n²)"]
ops_n2 = 100_000 ** 2          # 100억 → 1초 안에 불가능
complexity = {
    "list에서 x in lst":      "O(n)",
    "set에서 x in s":         "O(1)",
    "정렬":                   "O(n log n)",
    "정렬된 리스트에서 bisect": "O(log n)",
    "heapq.heappush":         "O(log n)",
}
```
</details>

In [ ]:
check("복잡도 순서", lambda: order == ["O(1)", "O(log n)", "O(n)", "O(n log n)", "O(n²)"])
check("100억", lambda: ops_n2 == 10_000_000_000)
check("연산별 복잡도", lambda: list(complexity.values()) == ["O(n)", "O(1)", "O(n log n)", "O(log n)", "O(log n)"])

## 5-2. 리스트 vs set — "있나?" 확인은 해시로
같은 일을 하는 두 함수를 만들고 시간을 비교해요. `has_dup_slow`는 이중 반복(O(n²)), `has_dup_fast`는 set(O(n)).

In [ ]:
def has_dup_slow(nums):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j]:
                return True
    return False

def has_dup_fast(nums):
    ___

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def has_dup_slow(nums):
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            if nums[i] == nums[j]:
                return True
    return False

def has_dup_fast(nums):
    seen = set()
    for x in nums:
        if x in seen:
            return True
        seen.add(x)
    return False          # 한 줄로: return len(set(nums)) != len(nums)
```
</details>

In [ ]:
data = random.sample(range(10**7), 3000)               # 중복 없음 (최악의 경우)
t = time.perf_counter(); r1 = has_dup_slow(data); t_slow = time.perf_counter() - t
t = time.perf_counter(); r2 = has_dup_fast(data); t_fast = time.perf_counter() - t
print(f"slow {t_slow:.3f}s / fast {t_fast:.5f}s")
check("결과 같음", lambda: r1 is r2 is False)
check("중복 찾기", lambda: has_dup_fast([3, 1, 4, 1, 5]) is True)
check("set이 훨씬 빠름", lambda: r2 is False and t_fast * 20 < t_slow)

### Two Sum (dict 활용)
`nums`에서 합이 `target`인 두 수의 **인덱스**를 O(n)으로 찾아요. 지나온 값 → 인덱스를 dict에 저장해 두면 돼요.

In [ ]:
def two_sum(nums, target):
    seen = {}                 # 값 → 인덱스
    for i, x in enumerate(nums):
        ___

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def two_sum(nums, target):
    seen = {}
    for i, x in enumerate(nums):
        if target - x in seen:
            return [seen[target - x], i]
        seen[x] = i
```
</details>

In [ ]:
check("two_sum 1", lambda: two_sum([2, 7, 11, 15], 9) == [0, 1])
check("two_sum 2", lambda: two_sum([3, 2, 4], 6) == [1, 2])

## 5-3. 정렬 + 이분탐색 (DB 인덱스의 원리)
정렬된 주문 금액에서 `lo` 이상 `hi` 이하인 개수를 O(log n)으로 세요. (`WHERE amount BETWEEN lo AND hi`)
그리고 `bisect_left`를 직접 구현해 봐요.

In [ ]:
def count_between(sorted_vals, lo, hi):
    return ___                    # bisect_left / bisect_right 두 번

def my_bisect_left(a, x):         # x를 넣을 수 있는 가장 왼쪽 위치
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if ___:
            lo = mid + 1
        else:
            hi = mid
    return lo

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def count_between(sorted_vals, lo, hi):
    return bisect.bisect_right(sorted_vals, hi) - bisect.bisect_left(sorted_vals, lo)

def my_bisect_left(a, x):
    lo, hi = 0, len(a)
    while lo < hi:
        mid = (lo + hi) // 2
        if a[mid] < x:
            lo = mid + 1
        else:
            hi = mid
    return lo
```
</details>

In [ ]:
amounts = sorted(random.randint(1000, 100000) for _ in range(10000))
check("범위 개수", lambda: count_between(amounts, 20000, 30000) == sum(20000 <= a <= 30000 for a in amounts))
check("경계값 포함", lambda: count_between([1, 2, 2, 2, 3], 2, 2) == 3)
check("bisect_left 직접 구현", lambda: all(my_bisect_left(amounts, x) == bisect.bisect_left(amounts, x)
                                          for x in [0, 1000, 55555, amounts[500], 10**6]))

## 5-4. BFS — 가중치 없는 최단 거리 (큐 = deque)
`#`은 벽이에요. `S`에서 `E`까지 최소 몇 칸 이동해야 하나요? 못 가면 -1.

In [ ]:
def shortest(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = ___                      # 앞에서 꺼내기
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and ___:   # 처음 가는 칸이면
                dist[(nr, nc)] = ___
                q.append((nr, nc))
    return -1

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def shortest(grid):
    R, C = len(grid), len(grid[0])
    start = next((r, c) for r in range(R) for c in range(C) if grid[r][c] == "S")
    dist = {start: 0}
    q = deque([start])
    while q:
        r, c = q.popleft()
        if grid[r][c] == "E":
            return dist[(r, c)]
        for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            nr, nc = r + dr, c + dc
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] != "#" and (nr, nc) not in dist:
                dist[(nr, nc)] = dist[(r, c)] + 1
                q.append((nr, nc))
    return -1
```
</details>

In [ ]:
g1 = ["S.#.....",
      ".##.###.",
      "....#...",
      "##.##.#E"]
g2 = ["S#E",
      "##."]
check("최단 거리", lambda: shortest(g1) == 14)
check("못 가면 -1", lambda: shortest(g2) == -1)

## 5-5. 힙 — 상위 k개 (BM25 검색 결과 자르기)
문서 10만 개 점수 중 상위 5개를 뽑아요. 전부 정렬(O(n log n)) 대신 **크기 k인 최소 힙**(O(n log k))을 써요.
힙 맨 위(가장 작은 값)보다 큰 점수가 오면 교체해요.

In [ ]:
def top_k(scores, k):
    heap = []                                   # (점수, 문서id) 최소 힙
    for doc_id, s in enumerate(scores):
        if len(heap) < k:
            heapq.heappush(heap, (s, doc_id))
        elif ___:                               # 힙의 최솟값보다 크면
            heapq.heapreplace(heap, (s, doc_id))
    return [doc for s, doc in sorted(heap, reverse=True)]

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def top_k(scores, k):
    heap = []
    for doc_id, s in enumerate(scores):
        if len(heap) < k:
            heapq.heappush(heap, (s, doc_id))
        elif s > heap[0][0]:
            heapq.heapreplace(heap, (s, doc_id))
    return [doc for s, doc in sorted(heap, reverse=True)]
```
</details>

In [ ]:
scores = [random.random() for _ in range(100_000)]
expected = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]
check("상위 5개", lambda: top_k(scores, 5) == expected)
check("heapq.nlargest와 같음", lambda: top_k(scores, 5) == [i for _, i in heapq.nlargest(5, ((s, i) for i, s in enumerate(scores)))])

## 5-6. DP — 같은 부분 문제를 다시 풀지 않기
### 동전 최소 개수
`coins`로 `amount`를 만드는 최소 동전 수. 못 만들면 -1.
`dp[x]` = x원을 만드는 최소 동전 수 → `dp[x] = min(dp[x - c] + 1)`

In [ ]:
def min_coins(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    for x in range(1, amount + 1):
        for c in coins:
            if c <= x:
                dp[x] = ___
    return ___

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
def min_coins(coins, amount):
    INF = float("inf")
    dp = [0] + [INF] * amount
    for x in range(1, amount + 1):
        for c in coins:
            if c <= x:
                dp[x] = min(dp[x], dp[x - c] + 1)
    return dp[amount] if dp[amount] != INF else -1
```
</details>

In [ ]:
check("그리디로 안 되는 경우 (6 = 3+3)", lambda: min_coins([1, 3, 4], 6) == 2)
check("4200원", lambda: min_coins([10, 50, 100, 500, 1000], 4200) == 6)
check("불가능", lambda: min_coins([2], 3) == -1)

### 메모이제이션 체감: 피보나치
`functools.lru_cache`를 붙이면 같은 인자의 결과를 저장해요. 호출 횟수를 비교해요.

In [ ]:
from functools import lru_cache
calls = {"plain": 0, "memo": 0}

def fib(n):
    calls["plain"] += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

___                                  # 데코레이터 한 줄
def fib_memo(n):
    calls["memo"] += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

fib(25), fib_memo(25), calls

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
from functools import lru_cache
calls = {"plain": 0, "memo": 0}

def fib(n):
    calls["plain"] += 1
    return n if n < 2 else fib(n - 1) + fib(n - 2)

@lru_cache(maxsize=None)
def fib_memo(n):
    calls["memo"] += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

fib(25), fib_memo(25), calls
```
</details>

In [ ]:
check("값 같음", lambda: fib_memo(25) == 75025)
check("메모 버전은 26번만 호출", lambda: calls["memo"] == 26)
check("그냥 버전은 24만 번 넘게", lambda: calls["plain"] > 240_000)

## 5-7. 실무 연결: LRU 캐시 (해시 + 순서)
가장 오래 안 쓴 것부터 버리는 캐시예요. `OrderedDict`의 `move_to_end`와 `popitem(last=False)`를 써요.

In [ ]:
class LRU:
    def __init__(self, cap):
        self.cap, self.d = cap, OrderedDict()
    def get(self, k):
        if k not in self.d:
            return None
        ___                         # 방금 썼으니 맨 뒤로
        return self.d[k]
    def put(self, k, v):
        self.d[k] = v
        self.d.move_to_end(k)
        if len(self.d) > self.cap:
            ___                     # 가장 오래된 것(맨 앞) 버리기

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
class LRU:
    def __init__(self, cap):
        self.cap, self.d = cap, OrderedDict()
    def get(self, k):
        if k not in self.d:
            return None
        self.d.move_to_end(k)
        return self.d[k]
    def put(self, k, v):
        self.d[k] = v
        self.d.move_to_end(k)
        if len(self.d) > self.cap:
            self.d.popitem(last=False)
```
</details>

In [ ]:
c = LRU(2)
c.put("배송", 1); c.put("환불", 2)
c.get("배송")                    # 배송을 최근에 씀
c.put("교환", 3)                 # 넘침 → 가장 오래 안 쓴 '환불'이 빠져야 함
check("LRU 제거", lambda: c.get("환불") is None and c.get("배송") == 1 and c.get("교환") == 3)

In [ ]:
# 상황에 맞는 자료구조: "list" / "set" / "dict" / "deque" / "heapq"
pick = {
    "방문한 URL인지 빠르게 확인":            ___,
    "BFS 큐 (앞에서 꺼내기)":               ___,
    "계속 들어오는 작업 중 우선순위 최소 꺼내기": ___,
    "회원id → 회원정보 찾기":               ___,
    "순서대로 쌓고 인덱스로 접근":            ___,
}

<details><summary>💡 정답 보기 (먼저 직접 풀어보고 열기)</summary>

```python
pick = {
    "방문한 URL인지 빠르게 확인":            "set",
    "BFS 큐 (앞에서 꺼내기)":               "deque",
    "계속 들어오는 작업 중 우선순위 최소 꺼내기": "heapq",
    "회원id → 회원정보 찾기":               "dict",
    "순서대로 쌓고 인덱스로 접근":            "list",
}
```
</details>

In [ ]:
check("자료구조 고르기", lambda: list(pick.values()) == ["set", "deque", "heapq", "dict", "list"])